# Generator viewer

**One generator, one sample, in detail.** Paste the `generator` and `filtering`
blocks of a run config, draw a single sample, apply the filters, and look at the
result. Change `SEED` and re-run to see how the structure varies from draw to
draw.

For a quick look at what every generator produces, use
`generators_overview.ipynb` instead.

- 2D samples are shown as an image.
- 3D samples are shown as an interactive surface of one phase (drag to rotate),
  plus three mid-plane slices.

The interactive 3D view uses [Plotly](https://plotly.com/python/)
(`pip install plotly`). Without it, the notebook falls back to a static
Matplotlib view.

In [ ]:
import itertools
import json
import time

import matplotlib.pyplot as plt
import numpy as np
from skimage import measure

from compgen.runner import get_object_from_module, setup_generator

try:
    import plotly.graph_objects as go
except ImportError:
    go = None
    print("Plotly not found: 3D views will be static. Install it with `pip install plotly`.")

## Parameters

Paste the `"generator"` and `"filtering"` blocks of a run config (JSON) into
`CONFIG`; anything else in it is ignored. The generator is built and the filters
are applied exactly as in a run, including `refinement` (the sample is then
`2**refinement` times finer, as the run's filters see it). Leave `"filtering"`
empty, or drop it, to look at the raw sample.

`SEED` replaces any `"random_seed"` in the generator params: the same seed and
config always give the same structure.

`refinement` multiplies every axis by `2**refinement`, so the sample grows fast
in 3D: `N = [128, 128, 128]` with `refinement = 3` is 1024³ (a billion voxels).
Samples larger than `MAX_VOXELS` are refused before generating; set
`REFINEMENT = 0` (or lower it) to look at the same structure on a coarser grid.

In [ ]:
SEED = None  # change to explore randomness
REFINEMENT = None  # None: use the config's; e.g. 0 to view a coarser sample quickly
MAX_VOXELS = 64_000_000  # refuse larger samples (1024**3 would need tens of GB)

CONFIG = """
{
    "generator": {
        "type": "gen.SpinodalGenerator",
        "params": {
            "N": [64, 64, 64],
            "refinement": 2,
            "vf": [0.1, 0.9],
            "l": [0.4, 2.0],
            "bandwidth": [0.25, 0.75]
        }
    },
    "filtering": [
        {"type": "filters.mean_filter", "params": {"radius": 0.02, "method": "fft"}},
        {"type": "filters.defeature", "params": {"min_size": 0.02}},
        {"type": "filters.make_periodic", "params": {"margin_cells": 11, "shift": true, "center": true}}
    ]
}
"""

# Other generators, e.g.:
#   {"type": "gen.VoronoiGenerator",
#    "params": {"N": [64, 64, 64], "grain_size": 0.35, "thickness": 0.12, "mode": "faces", "invert": false}}
#   {"type": "gen.ParticleGenerator",
#    "params": {"N": [64, 64, 64], "vf": 0.2, "kind": "spheres", "shape_params": {"size_m": 0.15}}}

## Sample and filter

In [ ]:
config = json.loads(CONFIG)
generator_spec = config["generator"]
params = generator_spec.setdefault("params", {})
params["random_seed"] = SEED
if REFINEMENT is not None:
    params["refinement"] = REFINEMENT
if "N" in params:
    shape = [int(n * 2 ** params.get("refinement", 0)) for n in params["N"]]
    if np.prod(shape, dtype=float) > MAX_VOXELS:
        raise ValueError(
            f"The sample would be {' x '.join(map(str, shape))} = {np.prod(shape, dtype=float):.3g} voxels "
            f"(N = {params['N']}, refinement = {params.get('refinement', 0)}), above MAX_VOXELS = "
            f"{MAX_VOXELS:.3g}. Set REFINEMENT = 0 (or lower), reduce N, or raise MAX_VOXELS."
        )
generator = setup_generator(generator_spec)



def param_name(gen, value):
    """Name of the generator attribute holding `value` (an `x_input`, or a dict entry)."""
    attrs = sorted(vars(gen).items(), key=lambda kv: not kv[0].endswith("_input"))
    for key, attr in attrs:
        if attr is value:
            return key.removesuffix("_input")
    for key, attr in attrs:
        if isinstance(attr, dict):
            for sub, item in attr.items():
                if item is value:
                    return f"{key.removesuffix('_input')}.{sub}"
    return "?"


# Record every value the generator draws (ranges are sampled inside sample());
# the draws themselves are unchanged, so the structure is the same.
drawn = []
draw = generator._sample_value


def recording_draw(value):
    result = draw(value)
    drawn.append((param_name(generator, value), value, result))
    return result


generator._sample_value = recording_draw
start = time.perf_counter()
raw = generator.sample()
generator._sample_value = draw
print(f"{generator_spec['type']}: shape {raw.shape}, True fraction {raw.mean():.3f} "
      f"({time.perf_counter() - start:.2f} s)")
for name, given, value in drawn:
    if isinstance(given, list) and len(given) == 2:
        print(f"  {name} = {value:.6g}  (drawn from {given})")
    else:
        print(f"  {name} = {value}  (fixed)")

structure = raw
for spec in config.get("filtering", []):
    start = time.perf_counter()
    structure = get_object_from_module(spec["type"])(structure, **spec.get("params", {}))
    print(f"  after {spec['type']}: shape {structure.shape}, True fraction {structure.mean():.3f} "
          f"({time.perf_counter() - start:.2f} s)")

### What the filters changed

Mid-plane (3D) or full (2D) image before and after filtering, and the pixels
that changed phase. A filter may also shift the whole structure periodically
(`make_periodic` with `"shift": true`); the filtered image is shifted back
before comparing, so only real changes are shown. Skipped when there are no
filters, or when a filter changes the shape (e.g. `make_orthogonal`).

In [ ]:
def mid_plane(img):
    return img if img.ndim == 2 else img[:, :, img.shape[2] // 2]


def periodic_shift(a, b):
    """Circular shift s with np.roll(a, s) closest to b (FFT cross-correlation)."""
    fa, fb = np.fft.fftn(np.where(a, 1.0, -1.0)), np.fft.fftn(np.where(b, 1.0, -1.0))
    corr = np.fft.ifftn(fb * np.conj(fa)).real
    return np.unravel_index(np.argmax(corr), corr.shape)


if not config.get("filtering"):
    print("No filters in CONFIG.")
elif structure.shape != raw.shape:
    print(f"The filters changed the shape {raw.shape} -> {structure.shape}; no comparison.")
else:
    shift = periodic_shift(raw, structure)
    aligned = np.roll(structure, [-int(s) for s in shift], axis=tuple(range(raw.ndim)))
    if any(shift):
        print(f"The filters shifted the structure periodically by {tuple(int(s) for s in shift)} voxels; "
              "it is shifted back below.")
    before, after = mid_plane(raw), mid_plane(aligned)
    changed = np.ones(before.shape + (3,))
    changed[before & ~after] = (0.85, 0.2, 0.15)   # True -> False
    changed[~before & after] = (0.15, 0.45, 0.85)  # False -> True
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
    for ax, img, title in ((axes[0], before, "raw sample"), (axes[1], after, "filtered")):
        ax.imshow(img.T, cmap="gray", origin="lower", interpolation="nearest")
        ax.set_title(title)
    axes[2].imshow(np.transpose(changed, (1, 0, 2)), origin="lower", interpolation="nearest")
    axes[2].set_title(f"changed: {np.mean(raw != aligned):.2%} of all voxels\n"
                      "red True -> False, blue False -> True", fontsize=10)
    for ax in axes:
        ax.axis("off")
    fig.suptitle("white = True" + ("" if raw.ndim == 2 else ", z = 1/2 slice"))
    plt.show()

## View

`PHASE` picks the phase drawn as a surface in 3D (`None`: the minority phase,
e.g. the boundaries of a Voronoi structure with `invert=False`). For large `N`,
raise `STEP` to coarsen the surface and keep the view responsive.

Unit-cell edges are outlined in every view. Set `TILE = 2` to draw 2 periodic
copies per axis and check that the structure continues seamlessly across the
cell faces (in 3D this makes the surface 8 times larger, so raise `STEP` too).

In [ ]:
PHASE = None  # True, False, or None for the minority phase
STEP = 1      # marching-cubes step size (2 or 3 for N >= 128)
TILE = 1      # periodic copies per axis (2 shows how the structure repeats)
CELL_COLOR = "#e0461f"  # unit-cell outlines


def tiled(img):
    return np.tile(img, (TILE,) * img.ndim)


def cell_segments(dim):
    """End points of the unit-cell edges of a TILE**dim block, in cell units."""
    ticks = range(TILE + 1)
    segments = []
    for axis in range(dim):
        others = [a for a in range(dim) if a != axis]
        for fixed in itertools.product(ticks, repeat=dim - 1):
            a, b = np.zeros(dim), np.zeros(dim)
            a[others], b[others] = fixed, fixed
            b[axis] = TILE
            segments.append((a, b))
    return segments


def outline_2d(ax):
    for a, b in cell_segments(2):
        ax.plot(*zip(a, b), color=CELL_COLOR, lw=1.5)


def show_2d(img):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(tiled(img).T, cmap="gray", origin="lower", interpolation="nearest",
              extent=(0, TILE, 0, TILE))
    outline_2d(ax)
    ax.set_title(f"{type(generator).__name__}, seed {SEED}  (white = True)")
    ax.axis("off")
    plt.show()


def surface(volume, phase, step):
    """Triangle mesh of one phase, in unit-cell coordinates. The volume is padded
    with the other phase so the solids are closed where they cross the cell faces."""
    solid = tiled(volume if phase else ~volume)
    padded = np.pad(solid, 1, constant_values=False).astype(float)
    verts, faces, _, _ = measure.marching_cubes(padded, 0.5, step_size=step)
    return (verts - 1) / np.array(volume.shape), faces


def show_3d_interactive(verts, faces, phase):
    x, y, z = verts.T
    i, j, k = faces.T
    mesh = go.Mesh3d(
        x=x, y=y, z=z, i=i, j=j, k=k,
        color="#3b6fd8",
        lighting=dict(ambient=0.5, diffuse=0.8, specular=0.1, roughness=0.6),
        hoverinfo="skip",
    )
    lines = np.array([p for a, b in cell_segments(3) for p in (a, b, [np.nan] * 3)])
    cells = go.Scatter3d(x=lines[:, 0], y=lines[:, 1], z=lines[:, 2], mode="lines",
                         line=dict(color=CELL_COLOR, width=4), hoverinfo="skip")
    axis = dict(range=[-0.01, TILE + 0.01], visible=False)
    fig = go.Figure([mesh, cells])
    fig.update_layout(
        title=f"{type(generator).__name__}, seed {SEED}: phase {phase} "
        f"(fraction {np.mean(structure == phase):.3f})",
        scene=dict(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube",
                   camera=dict(eye=dict(x=1.6, y=1.4, z=1.1))),
        width=700, height=650, margin=dict(l=0, r=0, t=50, b=0),
        dragmode="orbit", showlegend=False,
    )
    fig.show()


def show_3d_static(verts, faces, phase):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(projection="3d")
    ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                    color="tab:blue", linewidth=0, antialiased=False, shade=True)
    for a, b in cell_segments(3):
        ax.plot(*zip(a, b), color=CELL_COLOR, lw=1.5)
    ax.set(xlim=(0, TILE), ylim=(0, TILE), zlim=(0, TILE), xticks=[], yticks=[], zticks=[])
    ax.set_box_aspect((1, 1, 1))
    ax.set_title(f"{type(generator).__name__}, seed {SEED}: phase {phase}")
    plt.show()


def show_slices(volume):
    mid = [s // 2 for s in volume.shape]
    slices = {"x = 1/2": volume[mid[0]], "y = 1/2": volume[:, mid[1]], "z = 1/2": volume[:, :, mid[2]]}
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
    for ax, (name, sl) in zip(axes, slices.items()):
        ax.imshow(tiled(sl).T, cmap="gray", origin="lower", interpolation="nearest",
                  extent=(0, TILE, 0, TILE))
        outline_2d(ax)
        ax.set_title(name)
        ax.axis("off")
    fig.suptitle("mid-plane slices (white = True)")
    plt.show()


if structure.ndim == 2:
    show_2d(structure)
else:
    phase = (structure.mean() <= 0.5) if PHASE is None else PHASE
    verts, faces = surface(structure, phase, STEP)
    print(f"surface: {len(faces):,} triangles")
    if go is not None:
        show_3d_interactive(verts, faces, phase)
    else:
        show_3d_static(verts, faces, phase)
    show_slices(structure)